In [ ]:
# ------------------------------------------------------------------------
# RF-DETR
# Copyright (c) 2025 Roboflow. All Rights Reserved.
# Licensed under the Apache License, Version 2.0 [see LICENSE for details]
# ------------------------------------------------------------------------

# Train one model on RF100-VL sources with size-balanced batch weights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/docs/cookbooks/multi-source-rf100vl.ipynb)

[RF100-VL](https://github.com/roboflow/rf100-vl) is a benchmark of 100 small, independently-collected
detection datasets spanning aerial, industrial, flora/fauna, sport, document and lab-imaging domains -
used elsewhere in these docs to *evaluate* generalization (see [Benchmarks](../learn/benchmarks.md)), one
fine-tune per dataset. This cookbook instead uses three of them as an example of *joint* multi-dataset
training: one `RFDETRSmall` fine-tuned on all three at once with `WeightedMultiSourceBatchSampler`.

The three sources differ by more than 20x in size - `wheel-defect-detection` (309 images),
`aerial-cows` (1,107), `bees` (6,640) - so a plain `ConcatDataset` would let `bees` dominate every batch
while the industrial defect set contributes almost nothing to the gradient. Instead of hand-picking
weights (as the [Universe cookbook](multi-source-batch-sampler/) does), this notebook computes them
automatically as the **inverse of each source's measured train-split size**, so all three sources get a
comparable share of every batch regardless of how their sizes compare.

See [Training customization -> Mixing datasets](../learn/train/customization.md) for the sampler API
reference.

## Setup

Install `rfdetr` with the `train` extra (PyTorch Lightning, COCO eval) plus `roboflow` for the
RF100-VL download. A GPU is recommended; drop `BATCH_SIZE` if you hit out-of-memory.

In [ ]:
!pip install -q "rfdetr[train]>=1.10.0" roboflow

In [ ]:
"""Train one RF-DETR model on three RF100-VL sources with inverse-size-weighted batches."""

import json
import os
from collections import Counter
from pathlib import Path
from typing import Any

from roboflow import Roboflow
from torch.utils.data import ConcatDataset

from rfdetr import RFDETRSmall
from rfdetr.config import TrainConfig
from rfdetr.datasets import WeightedMultiSourceBatchSampler, compute_source_batch_sizes
from rfdetr.datasets.coco import (
    CocoDetection,
    annotated_category_ids,
    filter_parent_categories,
    make_coco_transforms,
)
from rfdetr.datasets.kornia_transforms import is_gpu_postprocess, resolve_backend_for_build
from rfdetr.training import RFDETRDataModule, RFDETRModelModule, build_trainer
from rfdetr.utilities.reproducibility import seed_all

In [ ]:
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"
OUTPUT_DIR = PROJECT_ROOT / "output" / "multi_source_rf100vl"

# All three live under the shared "rf100-vl" Roboflow workspace (github.com/roboflow/rf100-vl,
# rf100vl.roboflow100vl.get_rf100vl_projects), spanning three very different domains and sizes on
# purpose - that size spread is what makes inverse-size weighting worth demonstrating.
RF100VL_WORKSPACE = "rf100-vl"
SOURCES: list[dict[str, Any]] = [
    {
        "key": "wheel_defect",
        "name": "Wheel Defect Detection",
        "project": "wheel-defect-detection-e53jb-38chk-ytwg",
        "source_url": "https://universe.roboflow.com/rf100-vl/wheel-defect-detection-e53jb-38chk-ytwg",
    },
    {
        "key": "aerial_cows",
        "name": "Aerial Cows",
        "project": "aerial-cows-kt2wd-3jxcj-uvfx",
        "source_url": "https://universe.roboflow.com/rf100-vl/aerial-cows-kt2wd-3jxcj-uvfx",
    },
    {
        "key": "bees",
        "name": "Bees",
        "project": "bees-jt5in-6pbpl-aplx",
        "source_url": "https://universe.roboflow.com/rf100-vl/bees-jt5in-6pbpl-aplx",
    },
]

SEED = 7
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_WORKERS = 2
RESOLUTION = 512
LR = 1e-4
LR_ENCODER = 1e-4

print(f"sources={[source['key'] for source in SOURCES]}")

## 1 - Download the datasets

Get a free Roboflow API key at `app.roboflow.com/settings/api` and expose it as `ROBOFLOW_API_KEY`
(an environment variable locally, or a Colab secret with the same name). Each download is idempotent -
re-running skips the transfer if the target directory already exists.

The dataset version is resolved to the highest available one at download time rather than pinned to a
number, since a maintainer can publish a new RF100-VL export version at any point.

In [ ]:
try:
    from google.colab import userdata

    try:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY") or ""
    except Exception:
        ROBOFLOW_API_KEY = ""
except ImportError:
    ROBOFLOW_API_KEY = ""

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
if not ROBOFLOW_API_KEY:
    raise RuntimeError(
        "ROBOFLOW_API_KEY not found. "
        "In Colab: add it via Secrets (key icon). "
        "Locally: set the environment variable before running."
    )

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace(RF100VL_WORKSPACE)
for source in SOURCES:
    location = DATASETS_DIR / str(source["key"])
    project = workspace.project(str(source["project"]))
    # v.version is the plain trailing version number as a string (Roboflow's own v.id is the full
    # "workspace/project/N" path, which sorts wrong across multi-digit version boundaries, e.g. "9" > "10").
    latest_version = max(project.versions(), key=lambda version: int(version.version))
    dataset = latest_version.download("coco", location=str(location))
    source["dataset_dir"] = Path(dataset.location)
    source["train_ann"] = source["dataset_dir"] / "train" / "_annotations.coco.json"
    print(f"{source['key']}: version={latest_version.version} dir={source['dataset_dir']}")

## 2 - Unify class names

Each RF100-VL export remaps categories to its own `0..N-1` independently, and Roboflow prepends a
synthetic grouping category that RF-DETR drops. Concatenating the datasets as-is would mix those label
spaces. Build one class list (first-seen name wins) and a per-source `category_id -> label` map so every
source writes into the same head - here the three sources happen to contribute disjoint classes, but the
same union step is required whenever two sources could share a name.

In [ ]:
def _kept_categories(ann_path: Path) -> list[dict[str, Any]]:
    with ann_path.open() as handle:
        coco = json.load(handle)
    return filter_parent_categories(coco["categories"], annotated_category_ids(coco))


CLASS_NAMES: list[str] = []
name_to_index: dict[str, int] = {}
for source in SOURCES:
    source["cat2label"] = {}
    for category in _kept_categories(Path(source["train_ann"])):
        name = str(category["name"])
        key = name.casefold()
        if key not in name_to_index:
            name_to_index[key] = len(CLASS_NAMES)
            CLASS_NAMES.append(name)
        source["cat2label"][int(category["id"])] = name_to_index[key]

NUM_CLASSES = len(CLASS_NAMES)
print(f"num_classes={NUM_CLASSES}")
print(f"class_names={CLASS_NAMES}")
for source in SOURCES:
    names = [CLASS_NAMES[index] for index in sorted(set(source["cat2label"].values()))]
    print(f"{source['key']}: {names}")

## 3 - Compute inverse-size weights

`WeightedMultiSourceBatchSampler` takes relative weights, not raw sizes - passing the sizes themselves
back in would just reproduce plain `ConcatDataset` sampling, the behavior this sampler exists to avoid.
Weighting by `1 / size` instead gives every source a comparable share of each batch: `wheel_defect`
(309 images) ends up requesting roughly as many slots as `bees` (6,640 images), 21x larger.

In [ ]:
def _count_images(ann_path: Path) -> int:
    with ann_path.open() as handle:
        return len(json.load(handle)["images"])


for source in SOURCES:
    source["train_size"] = _count_images(Path(source["train_ann"]))

inverse_sizes = [1.0 / source["train_size"] for source in SOURCES]
total_inverse = sum(inverse_sizes)
WEIGHTS = [value / total_inverse for value in inverse_sizes]

for source, weight in zip(SOURCES, WEIGHTS, strict=True):
    print(f"{source['key']}: train_size={source['train_size']} weight={weight:.3f}")
print(f"slots_per_batch={compute_source_batch_sizes(BATCH_SIZE, WEIGHTS)}")

## 4 - Subclass `RFDETRDataModule`

`setup` concatenates one `CocoDetection` per source. The `build_train_sampler` hook then returns a
`WeightedMultiSourceBatchSampler` instead of a size-proportional shuffle, and RF-DETR's own
`train_dataloader` builds the DataLoader around it - so the loader keeps its configured `collate_fn`,
worker count, pinning and per-worker augmentation seeding without restating any of them.
`RFDETRModelModule.on_train_epoch_start` forwards `set_epoch` to it automatically every epoch, so no
extra callback is needed to advance the within-source shuffle. `epoch_length="smallest"` keeps this
demo short: the epoch ends once the smallest source (`wheel_defect`) has been seen once, so the larger
sources are sub-sampled rather than the small one being recycled many times.

> **Note:** Under DDP pass `use_distributed_sampler=False` to `build_trainer`. The sampler shards
> batches itself via `num_replicas` / `rank`; without this flag the run fails with a `RuntimeError`
> naming the fix.

In [ ]:
_SPLIT_DIRS = {
    "train": "train",
    "val": "valid",
    "test": "test",
}


def _build_source(
    source: dict[str, Any],
    image_set: str,
    *,
    resolution: int,
    model_config: Any,
    train_config: TrainConfig,
) -> CocoDetection:
    root = Path(source["dataset_dir"])
    split_dir = root / _SPLIT_DIRS[image_set]
    gpu_postprocess = is_gpu_postprocess(resolve_backend_for_build(train_config.augmentation_backend))
    return CocoDetection(
        split_dir,
        split_dir / "_annotations.coco.json",
        transforms=make_coco_transforms(
            image_set,
            resolution,
            multi_scale=train_config.multi_scale,
            expanded_scales=train_config.expanded_scales,
            skip_random_resize=not train_config.do_random_resize_via_padding,
            patch_size=model_config.patch_size,
            num_windows=model_config.num_windows,
            aug_config=train_config.aug_config,
            scale_jitter=train_config.scale_jitter,
            gpu_postprocess=gpu_postprocess,
        ),
        remap_category_ids=True,
        cat2label=source["cat2label"],
    )


class MultiSourceDataModule(RFDETRDataModule):
    """Train on several RF100-VL sources with inverse-size-weighted batches."""

    def __init__(
        self,
        model_config: Any,
        train_config: TrainConfig,
        sources: list[dict[str, Any]],
        weights: list[float],
    ) -> None:
        super().__init__(model_config, train_config)
        self.sources = sources
        self.weights = weights

    def setup(self, stage: str) -> None:
        super().setup(stage)
        if stage != "fit":
            return
        train_sources = [
            _build_source(
                source,
                "train",
                resolution=self.model_config.resolution,
                model_config=self.model_config,
                train_config=self.train_config,
            )
            for source in self.sources
        ]
        self._dataset_train = ConcatDataset(train_sources)
        self._source_sizes = [len(dataset) for dataset in train_sources]
        print(
            "source_sizes="
            + ", ".join(
                f"{source['key']}={size}" for source, size in zip(self.sources, self._source_sizes, strict=True)
            )
        )

    def build_train_sampler(self, dataset: Any) -> WeightedMultiSourceBatchSampler:
        """Return the batch sampler; the base ``train_dataloader`` builds the DataLoader around it."""
        world_size = self.trainer.world_size if self.trainer else 1
        rank = self.trainer.global_rank if self.trainer else 0
        return WeightedMultiSourceBatchSampler.from_concat_dataset(
            dataset,
            weights=self.weights,
            batch_size=self._resolve_batch_size(),
            num_replicas=world_size,
            rank=rank,
            seed=SEED,
            epoch_length="smallest",
            # Keep every gradient-accumulation window complete: the hook skips the dataset padding the
            # default path relies on, so the sampler has to round its own epoch to a whole number of windows.
            batch_multiple=self.train_config.grad_accum_steps,
        )

## 5 - Inspect one epoch of batches

Before training, iterate the sampler and count how many indices came from each source. Every batch
should match `compute_source_batch_sizes`. A size-proportional `ConcatDataset` shuffle would instead
follow the source-size ratios printed below - `bees` would dominate nearly every batch.

In [ ]:
seed_all(SEED)
variant = RFDETRSmall(  # type: ignore[no-untyped-call]
    num_classes=NUM_CLASSES,
    resolution=RESOLUTION,
)
variant.model_config.model_name = type(variant).__name__

PRIMARY_DIR = Path(SOURCES[0]["dataset_dir"])
train_config = TrainConfig(
    dataset_file="roboflow",
    dataset_dir=str(PRIMARY_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    lr=LR,
    lr_encoder=LR_ENCODER,
    warmup_epochs=1,
    use_ema=False,
    run_test=False,
    multi_scale=False,
    expanded_scales=False,
    tensorboard=False,
    wandb=False,
    mlflow=False,
    clearml=False,
    class_names=CLASS_NAMES,
    progress_bar="tqdm",
)

datamodule = MultiSourceDataModule(variant.model_config, train_config, SOURCES, WEIGHTS)
datamodule.setup("fit")
datamodule.train_dataloader()
sampler = datamodule.train_batch_sampler
assert sampler is not None

expected_slots = tuple(compute_source_batch_sizes(BATCH_SIZE, WEIGHTS))
source_sizes = sampler.source_sizes
total_images = sum(source_sizes)
print(f"expected_slots_per_batch={list(expected_slots)}")
print(f"size_proportional_share={[round(size / total_images, 3) for size in source_sizes]}")
print(f"batches_this_epoch={len(sampler)}")

header = f"{'batch':>5}  " + "  ".join(f"{str(source['key']):>16}" for source in SOURCES)
print(header)
compositions: list[tuple[int, ...]] = []
drawn: Counter[int] = Counter()
for batch_index, batch in enumerate(sampler):
    counts = [0] * len(source_sizes)
    for index in batch:
        start = 0
        for source_index, size in enumerate(source_sizes):
            if start <= index < start + size:
                counts[source_index] += 1
                drawn[source_index] += 1
                break
            start += size
    composition = tuple(counts)
    compositions.append(composition)
    if batch_index < 8:
        print(f"{batch_index:>5}  " + "  ".join(f"{count:>16}" for count in composition))
if len(compositions) > 8:
    print(f"... ({len(compositions) - 8} more batches)")

unique = set(compositions)
print(f"unique_batch_compositions={unique}")
print("samples_drawn_this_epoch=" + str({str(SOURCES[index]["key"]): drawn[index] for index in range(len(SOURCES))}))
assert unique == {expected_slots}, unique

## 6 - Fine-tune

Validation is skipped here (`limit_val_batches=0`): the val split of one source does not cover classes
that only appear in the other two, so COCO eval on that split would be misleading. Raise `EPOCHS` and
restore validation once you have a held-out mix you actually want to score.

In [ ]:
model = RFDETRModelModule(variant.model_config, train_config)
trainer = build_trainer(
    train_config,
    variant.model_config,
    use_distributed_sampler=False,
    num_sanity_val_steps=0,
    limit_val_batches=0,
)
trainer.fit(model, datamodule=datamodule)

## What to try next

- Swap in a different trio of [RF100-VL](https://universe.roboflow.com/rf100-vl) projects - the
  workspace holds 100 of them across aerial, industrial, flora/fauna, sport, document and lab-imaging
  domains - and re-run from Step 1; the class-union and inverse-size-weighting steps need no changes.
- Compare against proportional (size-based) weights by setting
  `WEIGHTS = [source["train_size"] / total_images for source in SOURCES]` and re-running the inspection
  cell - `bees` will then claim most of every batch instead of a comparable share.
- Pass `epoch_length="largest"` to see every image of `bees` once per epoch; `wheel_defect` will then be
  recycled, and the sampler logs a warning past a 10x recycle factor.
- Full API: [`WeightedMultiSourceBatchSampler`](../reference/training.md) and
  [Mixing datasets with a fixed per-batch ratio](../learn/train/customization.md).